# 🛠️ Stage 4: Data Transformation & Feature Engineering

### **Objective**
Prepare clean datasets for downstream business analysis and Power BI reporting by:
1. Converting time columns to standard `datetime` objects.
2. Extracting foundational time features (`year`, `month`, `day_name`, `quarter`).
3. Standardizing text formats across categorical variables.
4. Engineering reusable delivery metrics (`delay_minutes`, `is_delayed`).
5. Exporting analytical CSVs to `../data/transformed/`.

In [1]:
import pandas as pd
import numpy as np
import os

# Create transformed data output directory
os.makedirs('../data/transformed', exist_ok=True)

# Load Cleaned Data
df_customers = pd.read_csv('../data/cleaned/cleaned_customers.csv')
df_orders = pd.read_csv('../data/cleaned/cleaned_orders.csv')
df_delivery = pd.read_csv('../data/cleaned/cleaned_delivery.csv')
df_inventory = pd.read_csv('../data/cleaned/cleaned_inventory.csv')
df_feedback = pd.read_csv('../data/cleaned/cleaned_feedback.csv')
df_marketing = pd.read_csv('../data/cleaned/cleaned_marketing.csv')

print("✅ Cleaned CSVs successfully loaded into environment!")

✅ Cleaned CSVs successfully loaded into environment!


## 1. Datetime Parsing & Time-Dimension Extractions

In [2]:
# Parse datetime columns using exact schema names
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'], errors='coerce')
df_delivery['promised_time'] = pd.to_datetime(df_delivery['promised_time'], errors='coerce')
df_delivery['actual_time'] = pd.to_datetime(df_delivery['actual_time'], errors='coerce')
df_inventory['date'] = pd.to_datetime(df_inventory['date'], errors='coerce')
df_feedback['feedback_date'] = pd.to_datetime(df_feedback['feedback_date'], errors='coerce')
df_marketing['date'] = pd.to_datetime(df_marketing['date'], errors='coerce')

# Extract Date Features on df_orders
df_orders['order_year'] = df_orders['order_date'].dt.year
df_orders['order_month'] = df_orders['order_date'].dt.month
df_orders['order_day_name'] = df_orders['order_date'].dt.day_name()
df_orders['order_quarter'] = df_orders['order_date'].dt.quarter

# Validation
print("--- ORDERS DATE FEATURES VALIDATION ---")
display(df_orders[['order_id', 'order_date', 'order_year', 'order_month', 'order_day_name', 'order_quarter']].head(3))

--- ORDERS DATE FEATURES VALIDATION ---


,order_id,order_date,order_year,order_month,order_day_name,order_quarter
0,1961864118,2024-07-17 08:34:01,2024,7,Wednesday,3
1,1549769649,2024-05-28 13:14:29,2024,5,Tuesday,2
2,9185164487,2024-09-23 13:07:12,2024,9,Monday,3


## 2. Text Standardization & Delivery Metric Engineering

In [3]:
# Text Standardization
if 'payment_method' in df_orders.columns:
    df_orders['payment_method'] = df_orders['payment_method'].str.strip().str.title()

if 'delivery_status' in df_delivery.columns:
    df_delivery['delivery_status'] = df_delivery['delivery_status'].str.strip().str.title()

# Reusable Metric: Actual Delay in Minutes (Actual - Promised)
df_delivery['delay_minutes'] = (
    (df_delivery['actual_time'] - df_delivery['promised_time']).dt.total_seconds() / 60.0
)

# Reusable Flag: Binary Indicator for SLA Breach (1 = Delayed, 0 = On Time)
df_delivery['is_delayed'] = (df_delivery['delay_minutes'] > 0).astype(int)

# Validation
print("--- DELIVERY METRICS VALIDATION ---")
display(df_delivery[['order_id', 'promised_time', 'actual_time', 'delay_minutes', 'is_delayed']].head(3))

--- DELIVERY METRICS VALIDATION ---


,order_id,promised_time,actual_time,delay_minutes,is_delayed
0,1549769649,2024-05-28 13:25:29,2024-05-28 13:27:29,2.0,1
1,9185164487,2024-09-23 13:25:12,2024-09-23 13:29:12,4.0,1
2,5427684290,2023-11-20 05:17:39,2023-11-20 05:18:39,1.0,1


## 3. Export Transformed Datasets

In [4]:
# Save Transformed Files
df_customers.to_csv('../data/transformed/transformed_customers.csv', index=False)
df_orders.to_csv('../data/transformed/transformed_orders.csv', index=False)
df_delivery.to_csv('../data/transformed/transformed_delivery.csv', index=False)
df_inventory.to_csv('../data/transformed/transformed_inventory.csv', index=False)
df_feedback.to_csv('../data/transformed/transformed_feedback.csv', index=False)
df_marketing.to_csv('../data/transformed/transformed_marketing.csv', index=False)

print("🚀 All transformed CSVs successfully saved to '../data/transformed/'!")

🚀 All transformed CSVs successfully saved to '../data/transformed/'!


### 💡 Transformation Summary

- Converted date/time columns to standard `datetime` data types across all datasets.
- Extracted core temporal dimensions (`order_year`, `order_month`, `order_day_name`, `order_quarter`) from `order_date`.
- Standardized categorical text formats (`payment_method`, `delivery_status`).
- Engineered core delay features (`delay_minutes`, `is_delayed`) to support delivery SLA analysis.
- Exported 6 cleaned and transformed datasets to `../data/transformed/` for downstream EDA and Power BI modeling.